# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets and their fields, showing each by their `@id` field and friendly label (if available).

-- If the dataset includes multiple record sets, their `@id`s will be printed; for each record set, fields and their `@id` are also shown. --

In [ ]:
# List available record sets and their fields by `@id`
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"--- RecordSet: {rs.id}")
    print(f"Label: {getattr(rs, 'label', None) or getattr(rs, 'name', None)}")
    print(f"Type: {getattr(rs, 'type', None)}")
    print("Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"  - @id: {field.id} | Label/Name: {getattr(field, 'label', None) or getattr(field, 'name', None)} | DataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: Extracting the first record set
dfs = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
for record_set_id in record_set_ids:
    # Load records as dictionaries
    records = list(dataset.records(record_set=record_set_id))
    # Build DataFrame
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records for record_set {record_set_id}")

if record_set_ids:
    example_rs = record_set_ids[0]
    print(f"Columns for record set {example_rs}:")
    print(dfs[example_rs].columns.tolist())
    display(dfs[example_rs].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping records.

In [ ]:
# Choose a numeric field from the DataFrame for EDA
import numpy as np

numeric_field = None
rs_id = None
# Try to automatically pick a numeric field
for r_id, df in dfs.items():
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            rs_id = r_id
            break
    if numeric_field:
        break

if numeric_field and rs_id:
    print(f"Using record set: {rs_id}")
    print(f"Using numeric field: {numeric_field}")
    df = dfs[rs_id]
    # Example criteria: Use values above the median as a threshold
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (median):\n", filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try to pick a group-by categorical field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields from the dataset.

This example creates a histogram of the selected numeric field and, if possible, a boxplot grouped by a selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and rs_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dfs[rs_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field:
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=dfs[rs_id][group_field], y=dfs[rs_id][numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset using `mlcroissant`, including:
- Inspecting available record sets and their fields by `@id`
- Loading data into DataFrames
- Filtering and normalizing a numeric field, and optionally grouping by a categorical field
- Visualizing field distributions

For a deeper analysis, you may consult the dataset's [FAIR^2 metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and documentation.